# Soup AI Engineer take-home: full run on Kaggle

**Kaggle:** Accelerator **GPU T4 x2**, **Internet on**. Training, merge, quantize, the masking check and
the kernels experiment use **GPU 0 only** (`CUDA_VISIBLE_DEVICES=0`, nvidia-smi `-i 0`); only verify can
use both GPUs (one process per GPU), set `PARALLEL_VERIFY = False` to keep it on GPU 0 too.
Private repo: Kaggle secret `GITHUB_TOKEN` (read-only); optional `HF_TOKEN`.
Final run: set `COMMIT` and `FINAL = True`, then **Save Version → Save & Run All**.

Order: setup → tests (`tests/`, `kernels/`) → prepare data + memory budget → plain chain (train, merge,
quantize, smoke generation) → task A (prompt masking) → verify on 20 rows → verify on 300 rows →
silent failures → language → kernels experiment (separate, never used for the verdict) → budget vs
measured → verdict → summary → archive. Every failed step stops the notebook (assert).
About 1–1.5 h. Disk: ~12 GB of artifacts in `/kaggle/working` (limit 20 GB).

In [ ]:
REPO_URL = "https://github.com/Adul23/soup-ai-tz.git"
COMMIT = ""            # the exact commit; empty = default branch (debug only)
FINAL = False          # True for the submitted run: COMMIT is then required
BASE_WEIGHTS = "fp16"  # what merge.py merges into: "fp16" (original weights) or "nf4-dequant"
PARALLEL_VERIFY = True # verify on GPU 0 + GPU 1 at once (needs T4 x2)
STOP_ON_CHECK_FAIL = True  # task A: a failed check stops the notebook (CLAUDE.md rule)
RUN = {"budget": True, "smoke": True, "check_a": True, "verify_quick": True, "verify_full": True,
       "silent_failures": True, "language": True, "kernels": True, "budget_compare": True, "verdict": True}
SKIP_SHIP = False     # silent_failures: True skips the (slow) soup ship runs

assert not FINAL or COMMIT, "final run: set COMMIT to the exact sha"
assert BASE_WEIGHTS in ("fp16", "nf4-dequant")

## Setup: GPU check, clone, pinned packages

In [ ]:
import base64, os, subprocess, sys, time
from pathlib import Path

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # CUDA device 0 == nvidia-smi GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4, even on Kaggle's T4 x2; set before anything touches CUDA
ON_KAGGLE = Path("/kaggle/working").exists()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
WORK = ROOT / f"run_{COMMIT[:8] or 'head'}_{time.strftime('%Y%m%dT%H%M%S')}"  # a fresh folder per Run All


def secret(name):
    """Kaggle or Colab secret, or "" if there is none. Never printed."""
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name) or ""
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def sh(cmd, cwd=None, env=None):
    """Run a shell command, stream its output into the cell, return the exit code."""
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, env=env,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")

# nvidia-smi ignores CUDA_VISIBLE_DEVICES and lists every GPU: always ask for GPU 0 explicitly
gpus = subprocess.run(["nvidia-smi", "-i", "0", "--query-gpu=name,memory.used", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("GPU 0:", gpus)
assert "T4" in gpus, "this run must be on a T4"
used = int(gpus.split(",")[1].split()[0])
assert used < 50, f"GPU 0 already holds {used} MiB: restart the session, do not touch CUDA before the run"

# clone: the token goes in a header for this one command, never in the URL, a file or the output
token = secret("GITHUB_TOKEN")
auth = ""
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = f"-c 'http.extraHeader=Authorization: Basic {basic}'"
code = subprocess.run(f"git {auth} clone -q {REPO_URL} {WORK}", shell=True,
                      capture_output=True, text=True)
err = code.stderr.replace(token, "***") if token else code.stderr
assert code.returncode == 0, "git clone failed: " + err
if COMMIT:
    assert sh(f"git checkout -q {COMMIT}", cwd=WORK) == 0, "git checkout failed"
sh("git log -1 --format='%H %s'", cwd=WORK)

assert sh(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=WORK) == 0, "pip install failed"
sh(f"{sys.executable} -m pip list 2>/dev/null | grep -iE '^(soup-cli|torch|torchao|transformers|peft|trl|bitsandbytes|accelerate|datasets) '")
# peft raises at adapter load (merge.py) when an old torchao is installed: fail here, not after training
assert sh(f"{sys.executable} -c 'from peft.import_utils import is_torchao_available as f; print(\"peft+torchao ok:\", f())'") == 0, \
    "peft rejects the installed torchao: see requirements.txt (torchao pin)"


N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
PARALLEL_VERIFY = PARALLEL_VERIFY and N_GPUS >= 2
print(f"GPUs: {N_GPUS}; verify on two GPUs: {PARALLEL_VERIFY}")

## Tests: `tests/` and `kernels/` (all must pass)

In [ ]:
assert sh(f"{sys.executable} -m pytest tests -q", cwd=WORK) == 0, "tests/ failed: stopping here"
assert sh(f"{sys.executable} -m pytest kernels -q", cwd=WORK) == 0, "kernels/ tests failed: stopping here"

## Memory budget (part 1), before any training

`task/prepare_data.py` first (it writes `data/data_report.json`; the budget takes the padded length from its
`tokens_full.train.p100`), then `memory/budget.py` → `runs/memory_budget.json`. Formulas only, nothing is loaded on the GPU.


In [ ]:
if RUN["budget"]:
    ts = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    assert sh(f"bash -c 'set -o pipefail; {sys.executable} task/prepare_data.py 2>&1 | tee logs/prepare_data_budget_{ts}.log'", cwd=WORK) == 0, "prepare_data failed"
    assert sh(f"{sys.executable} memory/budget.py", cwd=WORK) == 0, "memory/budget.py failed"


## Plain chain: prepare data, train, merge, quantize, smoke generation

`harness/smoke.py`: `run_train.py` (prepare data + plain `soup train`), `merge.py`, `quantize.py`,
`smoke_generate.py`; raw timestamped logs in `logs/`, summary in `runs/smoke.json`. ~15 min.

In [ ]:
RESUME_RUN_ID = ""  # set to the run id of a failed smoke run in this same WORK to continue it

if RUN["smoke"]:
    resume = f" --resume {RESUME_RUN_ID}" if RESUME_RUN_ID else ""
    code = sh(f"{sys.executable} harness/smoke.py --base-weights {BASE_WEIGHTS}{resume}", cwd=WORK)
    assert code == 0, f"smoke run failed (exit {code}); see runs/smoke.json and logs/"


## Task A: is the prompt masked in the labels?

`checks/tokenizer_template.py`: plain `soup train` up to the first `training_step`, the real batch
(input_ids, labels, attention_mask), counts over the whole tokenized training set.
Writes `runs/checks/prompt_masking.json`; exits 1 if the prompt is in the loss (then it also runs
`soup data doctor`, `soup train --dry-run`, `soup plan`, `soup ship` and keeps their output).

In [ ]:
if RUN["check_a"]:
    code = sh(f"{sys.executable} checks/tokenizer_template.py", cwd=WORK)
    print(f"exit code {code}")
    if STOP_ON_CHECK_FAIL:
        assert code == 0, "prompt masking check failed: see runs/checks/prompt_masking.json"

## Verify (part 2): base, zero_adapter, merged, base_4bit, merged_4bit

Controls are built first if missing (`verify/build_controls.py`). Two GPUs: `verify/run_parallel.py`
(GPU 0: base, zero_adapter [loss only], merged; GPU 1: base_4bit, merged_4bit) then
`verify/merge_verify.py`. One GPU: the same scripts one after the other on GPU 0.
First 20 held-out rows (also compares batch 1 vs batch 16 generation), then all 300.

In [ ]:
def verify(limit=None):
    lim = f" --limit {limit}" if limit else ""
    if PARALLEL_VERIFY:
        return sh(f"{sys.executable} verify/run_parallel.py --gen-batch 16{lim}", cwd=WORK)
    if not ((WORK / "artifacts/zero_merged").is_dir() and (WORK / "artifacts/base_4bit").is_dir()):
        code = sh(f"{sys.executable} verify/build_controls.py", cwd=WORK)
        if code:
            return code
    rid = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    code = sh(f"{sys.executable} verify/verify_artifacts.py --run-id {rid} --device 0 "
              f"--variants base,zero_adapter,merged,base_4bit,merged_4bit "
              f"--skip-gen-for zero_adapter --gen-batch 16{lim}", cwd=WORK)
    merged = sh(f"{sys.executable} verify/merge_verify.py --run-id {rid}", cwd=WORK)
    return code or merged

if RUN["verify_quick"]:
    code = verify(limit=20)
    assert code == 0, f"verify on 20 rows failed (exit {code}); see runs/verify.json and logs/"

In [ ]:
if RUN["verify_full"]:
    code = verify()
    assert code == 0, f"verify on 300 rows failed (exit {code}); see runs/verify.json and logs/"

## Silent failures (part 3): Soup's own checks and injected faults

`checks/silent_failures.py` → `runs/checks/silent_failures.json`: (a) `soup data doctor`, `soup train --dry-run`,
`soup ship` on the real data / config / adapter; (b) injected: pad = eos, max_length 64, duplicates and
"X және X" / ".Буква", renamed adapter modules, template without system prompt. For each: soup_caught,
our_check. Exit 1 = a Soup check failed on the real inputs, 3 = our check missed an injected fault (both are
findings: recorded, the run goes on to the verdict); 4 = a step crashed (stops the notebook). ~30–60 min with `soup ship` (`SKIP_SHIP = True` to leave it out).


In [ ]:
if RUN["silent_failures"]:
    skip = " --skip-ship" if SKIP_SHIP else ""
    code = sh(f"{sys.executable} checks/silent_failures.py{skip}", cwd=WORK)
    # 1 = a Soup check failed on the real inputs, 3 = our check missed an injected fault: findings,
    # recorded in the JSON and shown in the summary. Only 4 (a step crashed) stops the notebook.
    print(f"exit code {code}: " + {0: "done", 1: "FINDING: Soup check failed on the real inputs",
                                   3: "FINDING: our check missed an injected fault",
                                   4: "a step crashed"}.get(code, "unexpected exit code"))
    assert code in (0, 1, 3), f"silent failures: exit {code} (a step crashed or unexpected); see runs/checks/silent_failures.json"


## Language statistics of training data and generations

`checks/language.py` → `runs/checks/language.json`: loop share (3-gram > 2 times), "X және X" / "X, X",
Kyrgyz forms per 100 words, ".Буква" without a space; for train, held-out references and every variant's
greedy answers from the verify run in `runs/verify.json`.


In [ ]:
if RUN["language"]:
    assert sh(f"{sys.executable} checks/language.py", cwd=WORK) == 0, "checks/language.py failed"


## Memory experiment: ported kernels (NOT the main chain)

`harness/kernels_experiment.py`: `soup train` with `kernels.apply()` (fused head, starter-kit SwiGLU,
fp16 MLP), same config/seed/data, adapter → `artifacts/adapter_kernels`; merge → `merged_kernels`;
held-out loss via `verify_artifacts.py`; `runs/kernels_experiment.json`. Fails if a patch never engaged.
Runs after the full verify, so the plain merged loss it is compared with is on all 300 rows.

In [ ]:
if RUN["kernels"]:
    code = sh(f"CUDA_VISIBLE_DEVICES=0 {sys.executable} harness/kernels_experiment.py", cwd=WORK)
    assert code == 0, f"kernels experiment failed (exit {code}); see runs/kernels_experiment.json"

## Memory budget vs measured peaks

`memory/budget.py --compare`: the same budget plus the measured peaks from `runs/smoke.json` and
`runs/kernels_experiment.json` → `runs/memory_budget.json`.


In [ ]:
if RUN["budget_compare"]:
    assert sh(f"{sys.executable} memory/budget.py --compare", cwd=WORK) == 0, "memory/budget.py --compare failed"


## Verdict

`verdict.py` → `runs/verdict.json`, from `runs/verify.json`, `runs/checks/language.json`,
`runs/checks/silent_failures.json`, `runs/memory_budget.json`. Rules at the top of `verdict.py`.


In [ ]:
if RUN["verdict"]:
    assert sh(f"{sys.executable} verdict.py", cwd=WORK) == 0, "verdict.py failed (missing input?)"


## Summary (numbers from the JSON files only)

In [ ]:
import json

def load(rel_path):
    p = WORK / rel_path
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

dr = load("data/data_report.json")
if dr:
    print("== data: revision", dr["revision"], "| dropped", dr["dropped"]["total"], "| changed", dr["changed"]["total"])
    for split in dr["splits"]:
        print(f"  {split:12s} tokens_full {dr['tokens_full'][split]} | fertility {dr['fertility'][split]['answer_tokens_per_word']}"
              f" | would drop/change {dr['would_drop_or_change_not_applied'][split]} | round-trip {dr['round_trip'][split]}")

s = load("runs/smoke.json")
if s:
    print("== smoke:", "failed_step", s["failed_step"], "| commit", s["git"]["commit"], "| dirty", s["git"]["dirty"])
    for name, st in s["steps"].items():
        print(f"  {name:9s} exit {st['exit_code']}  wall {st['wall_s']} s  smi peak {st['nvidia_smi_peak_mib']} MiB  "
              f"torch alloc {st['torch_max_allocated_mib']} / reserved {st['torch_max_reserved_mib']} MiB")
    t = s["steps"].get("train", {})
    print(f"  train loss {t.get('first_loss')} -> {t.get('last_loss')}, steps {t.get('logged_steps')}/{t.get('max_steps')}")

a = load("runs/checks/prompt_masking.json")
if a:
    print("\n== task A: masked", a["masked"], "| prompt share in loss", a["prompt_share_in_loss"],
          "| header share", a["header_share_in_loss"], "| collator", a["collator"])
    print("  loss text of row 0:", a["evidence"]["loss_text"][:300])

v = load("runs/verify.json")
if v:
    print(f"\n== verify: rows {v['scored_rows']}/{v['heldout_rows']} (limit {v['limit']})")
    for name, e in v["variants"].items():
        b1 = e.get("batch1_vs_batch")
        print(f"  {name:13s} gpu{e['device']} loss {e['loss']:.5f} ppl {e['perplexity']:.3f} bpc {e['bits_per_char']:.4f} "
              f"eos {e.get('eos_share', '-')} len {e.get('mean_new_tokens', '-')} loop {e.get('loop_share', '-')}"
              + (f" b1≠b16 {b1['rows_differing']}/{b1['rows']}" if b1 else "") + f" | {e.get('generation', '')}")
    for pair, bs in v["paired_bootstrap"].items():
        print(f"  bootstrap {pair:24s} {bs['difference']:+.5f} [{bs['ci95'][0]:+.5f}, {bs['ci95'][1]:+.5f}]")
    c = v.get("nf4_codes_base_4bit_vs_merged_4bit")
    if c:
        print(f"  NF4 codes changed: targets {c['targets']['changed_share']}, non-targets {c['non_targets']['changed_share']}")
    print("  expectation_violated:", v["expectation_violated"])

k = load("runs/kernels_experiment.json")
if k:
    print("\n== kernels experiment: failed_step", k["failed_step"])
    if k.get("peaks"):
        for side, p in k["peaks"].items():
            print(f"  {side:8s} smi peak {p['nvidia_smi_peak_mib']} MiB, torch alloc {p['torch_max_allocated_mib']} / "
                  f"reserved {p['torch_max_reserved_mib']} MiB, wall {p['wall_s']} s")
        print("  loss curve:", k["loss_curve"])
        print("  held-out loss: kernels", k["heldout"]["kernels_merged"].get("loss"), "rows", k["heldout"]["kernels_scored_rows"],
              "| plain", k["heldout"]["plain_merged"].get("loss"), "rows", k["heldout"]["plain_scored_rows"])
        print("  counters:", k["counters"])

b = load("runs/memory_budget.json")
if b:
    print("\n== memory budget: plain", b["train_plain"]["torch_allocated_peak_mib"], "MiB, kernels",
          b["train_kernels"]["torch_allocated_peak_mib"], "MiB (torch allocated)")
    for name, c in (b.get("compare") or {}).items():
        print(f"  {name:14s} {c}")

sf = load("runs/checks/silent_failures.json")
if sf:
    print("\n== silent failures: crashed", sf.get("crashed"), "| our check missed", sf.get("our_check_missed"))
    for name, row in (sf.get("summary") or {}).items():
        print(f"  {name:20s} {row}")

lg = load("runs/checks/language.json")
if lg:
    print("\n== language (loop3 share, X,X share, kyrgyz/100 words, .Б share)")
    for name, g in lg["groups"].items():
        print(f"  {name:28s} {g['loop_share']:.3f} {g['repeat_pair_share']:.3f} "
              f"{g['kyrgyz_per_100_words_total']:.4f} {g['no_space_after_period_share']:.3f}")

vd = load("runs/verdict.json")
if vd:
    print("\n== VERDICT:", vd["verdict"])
    for name, r in vd["rules"].items():
        print(f"  {'ok  ' if r['ok'] else 'FAIL'} {name}: {r['rule']}")


## Archive

`logs/` (raw logs, nvidia-smi samples, generations, `pip freeze`), `runs/` (every JSON),
`data/data_report.json`, `AI_USAGE.md`, `CLAUDE.md` and the two LoRA adapters in one tar.gz in
`/kaggle/working`. The large merged / 4-bit models stay in `WORK/artifacts/`. With `FINAL = True` the
cell stops if the working tree is dirty or a recorded step ran on a dirty tree.


In [ ]:
ts = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
assert sh(f"{sys.executable} -m pip freeze > logs/pip_freeze_{ts}.txt", cwd=WORK) == 0, "pip freeze failed"

dirty = subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"], cwd=WORK,
                       capture_output=True, text=True).stdout.strip()
recorded_dirty = {p: (load(p) or {}).get("git", {}).get("dirty")
                  for p in ("runs/smoke.json", "runs/verify.json", "runs/verdict.json")}
print("git dirty now:", bool(dirty), "| recorded in", recorded_dirty)
if FINAL:
    assert not dirty, f"final run: the working tree is dirty:\n{dirty}"
    assert not any(recorded_dirty.values()), f"final run: a step ran on a dirty tree: {recorded_dirty}"

name = f"results_{WORK.name}.tar.gz"
keep = ["logs", "runs", "data/data_report.json", "AI_USAGE.md", "CLAUDE.md",
        "artifacts/adapter", "artifacts/adapter_kernels"]
parts = " ".join(p for p in keep if (WORK / p).exists())
print("archived:", parts)
assert sh(f"tar czf {ROOT / name} {parts}", cwd=WORK) == 0
print(ROOT / name, f"{(ROOT / name).stat().st_size / 2**20:.1f} MiB")
